In [1]:
from shapely.geometry import box, Point, LineString, Polygon, MultiPolygon
from shapely.ops import unary_union
import geopandas as gpd
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.plot import show
from rasterio.warp import calculate_default_transform, reproject
from rasterio.enums import Resampling
import osmnx as ox  
import contextily as ctx
from geopy.geocoders import Nominatim  
from geopy.extra.rate_limiter import RateLimiter
import os
#!pip install h3
import h3
import matplotlib.patches as mpatches

In [2]:
# Get the current working directory
current_path = os.getcwd()

# Define relative paths for data, output, and helper functions
data_in = os.path.join(current_path, "data")       # Folder with input data
output = os.path.join(current_path, "output")      # Folder for saving results

## Green spaces

In [3]:
file_name = 'data/OEFFGRUENFLOGD.json'
gdf_green = gpd.read_file(file_name)
print(gdf_green.columns)
print(len(gdf_green))
print(gdf_green['F_TYP'].value_counts())
gdf_green.head(3)

Index(['id', 'OBJECTID', 'TEXT13ID', 'F_TYP', 'T_TEXT', 'T_LANG', 'O', 'P',
       'O_UEREG', 'O_REGION', 'O_LOKAL', 'P_WLD', 'P_WAS', 'P_WIE', 'P_WGT',
       'P_LW', 'P_PK', 'P_GPK', 'P_GST', 'P_SGR', 'P_AGR', 'P_RGR', 'P_URB',
       'Q_MOB', 'Q_SPI', 'Q_GRI', 'Q_HUN', 'Q_BAB', 'Q_WC', 'Q_SPO', 'Q_TRB',
       'UMFANG', 'FLAECHE', 'SE_ANNO_CAD_DATA', 'geometry'],
      dtype='object')
1936
F_TYP
L    1629
U     171
R     136
Name: count, dtype: int64


,id,OBJECTID,TEXT13ID,F_TYP,T_TEXT,T_LANG,O,P,O_UEREG,O_REGION,...,Q_GRI,Q_HUN,Q_BAB,Q_WC,Q_SPO,Q_TRB,UMFANG,FLAECHE,SE_ANNO_CAD_DATA,geometry
0,OEFFGRUENFLOGD.2289,2289,Währinger Pa*,L,Währinger Park,Währinger Park,1,100000,0,0,...,0,1,0,1,1245,1,1460.941451,57020.131903,None,"MULTIPOLYGON (((16.35083 48.23369, 16.34929 48..."
1,OEFFGRUENFLOGD.2290,2290,Türkenschanz*,L,Türkenschanzplatz,Türkenschanzplatz,1,1,0,0,...,0,0,0,0,0,0,310.900196,2002.715993,None,"POLYGON ((16.32886 48.23482, 16.32824 48.23492..."
2,OEFFGRUENFLOGD.2291,2291,Kongreßpark,L,Kongreßpark/Kainzga,Kainzgasse,1,100000,0,0,...,0,1,0,2,12,1,249.284876,3243.484789,None,"POLYGON ((16.30871 48.22281, 16.30933 48.22285..."


In [4]:
print(gdf_green.crs)
gdf_green = gdf_green[['id', 'F_TYP', 'T_TEXT', 'FLAECHE', 'geometry']]
gdf_green = gdf_green.rename(columns={'FLAECHE': 'original_greenarea',
                                     'geometry': 'geometry_green'})
gdf_green.set_geometry('geometry_green', inplace=True)

gdf_green = gdf_green.to_crs(epsg=31256)
gdf_green['area_m2'] = gdf_green.geometry.area
print(gdf_green.crs)
gdf_green.head(3)

EPSG:4326
EPSG:31256


,id,F_TYP,T_TEXT,original_greenarea,geometry_green,area_m2
0,OEFFGRUENFLOGD.2289,L,Währinger Park,57020.131903,"MULTIPOLYGON (((1388.829 343923.309, 1274.719 ...",57020.220876
1,OEFFGRUENFLOGD.2290,L,Türkenschanzplatz,2002.715993,"POLYGON ((-242.971 344049.152, -288.821 344059...",2002.715999
2,OEFFGRUENFLOGD.2291,L,Kongreßpark/Kainzga,3243.484789,"POLYGON ((-1740.791 342714.09, -1694.181 34271...",3243.484798


In [5]:
print(len(gdf_green))
print("Sum green areas - from Flaeche:", gdf_green["original_greenarea"].sum())
print("Sum green areas - my calculation:", gdf_green['area_m2'].sum())

1936
Sum green areas - from Flaeche: 174781716.3472637
Sum green areas - my calculation: 177485054.0345264


## Census districts

In [6]:
boundaries_path = 'data/ZAEHLBEZIRKOGDPolygon.shp'
census_boundaries = gpd.read_file(boundaries_path)
print(census_boundaries.crs)
census_boundaries = census_boundaries.to_crs(epsg=31256)
census_boundaries.head(3)

EPSG:4326


,BEZNR,ZBEZNR,BEZ,ZBEZ,FLAECHE,UMFANG,AKT_TIMEST,SE_SDO_ROW,geometry
0,22.0,19.0,22,2219,431562.4878,2963.93,2025-05-10,305665.0,"POLYGON ((9534.781 343188.416, 9466.411 343046..."
1,19.0,9.0,19,1909,700939.3051,3670.56,2025-05-10,305666.0,"POLYGON ((419.711 344179.176, 419.451 344291.5..."
2,9.0,2.0,09,0902,471286.9574,3510.43,2025-05-10,305667.0,"POLYGON ((2280.931 343474.716, 2281.161 343474..."


In [7]:
census_boundaries = census_boundaries[['ZBEZ', 'FLAECHE', 'geometry']]
census_boundaries = census_boundaries.rename(columns={'FLAECHE': 'original_censusarea',
                                                     'geometry': 'geometry_census'})
census_boundaries.set_geometry('geometry_census', inplace=True)
census_boundaries['area_boundaries'] = census_boundaries.geometry.area
census_boundaries.head(3)

,ZBEZ,original_censusarea,geometry_census,area_boundaries
0,2219,431562.4878,"POLYGON ((9534.781 343188.416, 9466.411 343046...",431562.489038
1,1909,700939.3051,"POLYGON ((419.711 344179.176, 419.451 344291.5...",700939.307083
2,0902,471286.9574,"POLYGON ((2280.931 343474.716, 2281.161 343474...",471286.958700


In [8]:
print(len(census_boundaries))
print("Sum census district areas - from Flaeche:", census_boundaries["original_censusarea"].sum())
print("Sum census district areas - my calculation:", census_boundaries['area_boundaries'].sum())

250
Sum census district areas - from Flaeche: 414870565.6894
Sum census district areas - my calculation: 414870566.8331835


In [9]:
# green areas per district
print(census_boundaries.crs)
print(gdf_green.crs)

EPSG:31256
EPSG:31256


In [10]:
green_joined = gpd.sjoin(gdf_green, census_boundaries, how='inner', predicate='within')

green_joined['area_joined'] = green_joined.geometry.area
green_area_per_district = green_joined.groupby('ZBEZ')['area_joined'].sum().reset_index()
green_area_per_district['area_joined'] = green_area_per_district['area_joined'].fillna(0)

print("Sum green areas after join - my calculation:", green_area_per_district['area_joined'].sum())
print(len(green_area_per_district))

Sum green areas after join - my calculation: 25437072.41000665
230


In [13]:
green_in_census = gpd.overlay(census_boundaries, gdf_green, how='intersection')
#green_in_census = gpd.overlay(census_boundaries, gdf_green, how='intersection', keep_geom_type=False)
print(len(green_in_census))
print(green_in_census.isnull().sum())
green_in_census['new_area'] = green_in_census.geometry.area
print("Sum green areas after overlay - my calculation:", green_in_census["new_area"].sum())
print("Original green area:", gdf_green['area_m2'].sum())
print("Green area after overlay:", green_in_census['new_area'].sum())
green_in_census.head(5)

2414
ZBEZ                   0
original_censusarea    0
area_boundaries        0
id                     0
F_TYP                  0
T_TEXT                 0
original_greenarea     0
area_m2                0
geometry               0
dtype: int64
Sum green areas after overlay - my calculation: 126944693.16638353
Original green area: 177485054.0345264
Green area after overlay: 126944693.16638353


,ZBEZ,original_censusarea,area_boundaries,id,F_TYP,T_TEXT,original_greenarea,area_m2,geometry,new_area
0,2219,431562.4878,431562.489038,OEFFGRUENFLOGD.2676,L,Mühlgrund,76517.700862,76517.701080,"MULTIPOLYGON (((9103.751 342516.125, 9116.158 ...",247.320048
1,2219,431562.4878,431562.489038,OEFFGRUENFLOGD.2677,L,Hausgrun dweg,2739.814240,2739.814248,"POLYGON ((9466.411 343046.736, 9466.404 343046...",2737.393493
2,2219,431562.4878,431562.489038,OEFFGRUENFLOGD.2678,L,Vernholzgasse,5103.515597,5103.515612,"POLYGON ((8986.451 343400.036, 8995.931 343420...",158.497458
3,1909,700939.3051,700939.307083,OEFFGRUENFLOGD.2281,L,Linneplatz,5835.074062,5835.074079,"POLYGON ((487.739 344195.371, 427.179 344195.6...",5835.074079
4,1909,700939.3051,700939.307083,OEFFGRUENFLOGD.2462,L,Hugo-Wolf-Park,58333.397142,58333.397309,"POLYGON ((171.731 344507.126, 211.111 344507.3...",1.652203


In [17]:
# my groupby
green_in_census = green_in_census.sort_values(by=['ZBEZ', 'id']).reset_index(drop=True)
green_in_census.head(20)
green_in_census.groupby('id').size().reset_index(name='count').sort_values(by='count', ascending=False).head(20)
print(green_in_census[green_in_census['id']== 'OEFFGRUENFLOGD.3713']['new_area'].sum())
green_in_census[green_in_census['id']== 'OEFFGRUENFLOGD.3713']
# print(green_in_census.isnull().sum())

20297266.638403222
ZBEZ                   0
original_censusarea    0
area_boundaries        0
id                     0
F_TYP                  0
T_TEXT                 0
original_greenarea     0
area_m2                0
geometry               0
new_area               0
dtype: int64


In [11]:
# chatgpt groupby
# Total area of green space per district
green_area_per_district = (
    green_in_census.groupby('ZBEZ')['new_area']
    .sum()
    .reset_index()
)

# Merge with ALL census boundaries — keeps all 250
green_area_per_district = census_boundaries[['ZBEZ', 'area_boundaries']].merge(
    green_area_per_district, on='ZBEZ', how='left'
)

# Fill missing values (districts with no green space)
green_area_per_district['new_area'] = green_area_per_district['new_area'].fillna(0)
print(len(green_area_per_district))
print("Sum green areas per census district - chatgpt, after groupby:", green_area_per_district["new_area"].sum())
#green_area_per_district['green_ratio'] = green_area_per_district['new_area'] / green_area_per_district['area_boundaries']

250
Sum green areas per census district - chatgpt, after groupby: 126944693.16638353


In [18]:
# identifying lost green areas 
# green areas that has no intersect with census district boundaries at all
lost_green_ids = set(gdf_green['id']) - set(green_in_census['id'])
print(len(lost_green_ids))
print("Green areas fully outside all districts:", lost_green_ids)

green_clipped_per_id = green_in_census.groupby('id')['new_area'].sum().reset_index()
gdf_green['full_area'] = gdf_green.geometry.area
total_full_green_area = gdf_green['full_area'].sum()
green_area_comparison = gdf_green[['id', 'full_area']].merge(green_clipped_per_id, on='id', how='left')
green_area_comparison['new_area'] = green_area_comparison['new_area'].fillna(0)
green_area_comparison['loss_m2'] = green_area_comparison['full_area'] - green_area_comparison['new_area']
green_area_comparison['loss_pct'] = (green_area_comparison['loss_m2'] / green_area_comparison['full_area']) * 100

53
Green areas fully outside all districts: {'OEFFGRUENFLOGD.3864', 'OEFFGRUENFLOGD.3897', 'OEFFGRUENFLOGD.4010', 'OEFFGRUENFLOGD.3953', 'OEFFGRUENFLOGD.3912', 'OEFFGRUENFLOGD.3930', 'OEFFGRUENFLOGD.3854', 'OEFFGRUENFLOGD.3877', 'OEFFGRUENFLOGD.3853', 'OEFFGRUENFLOGD.3929', 'OEFFGRUENFLOGD.3867', 'OEFFGRUENFLOGD.3906', 'OEFFGRUENFLOGD.3905', 'OEFFGRUENFLOGD.3931', 'OEFFGRUENFLOGD.3874', 'OEFFGRUENFLOGD.3913', 'OEFFGRUENFLOGD.3860', 'OEFFGRUENFLOGD.3900', 'OEFFGRUENFLOGD.3898', 'OEFFGRUENFLOGD.3855', 'OEFFGRUENFLOGD.3961', 'OEFFGRUENFLOGD.3871', 'OEFFGRUENFLOGD.3907', 'OEFFGRUENFLOGD.3875', 'OEFFGRUENFLOGD.3872', 'OEFFGRUENFLOGD.3917', 'OEFFGRUENFLOGD.3863', 'OEFFGRUENFLOGD.4004', 'OEFFGRUENFLOGD.3920', 'OEFFGRUENFLOGD.3908', 'OEFFGRUENFLOGD.4020', 'OEFFGRUENFLOGD.4067', 'OEFFGRUENFLOGD.3993', 'OEFFGRUENFLOGD.3940', 'OEFFGRUENFLOGD.3919', 'OEFFGRUENFLOGD.4059', 'OEFFGRUENFLOGD.3858', 'OEFFGRUENFLOGD.3916', 'OEFFGRUENFLOGD.3876', 'OEFFGRUENFLOGD.3879', 'OEFFGRUENFLOGD.3859', 'OEFFGRUENFL

In [17]:
print(len(green_area_comparison))
green_area_comparison.head()

1936


,id,full_area,new_area,loss_m2,loss_pct
0,OEFFGRUENFLOGD.2289,57020.220876,57020.220876,7.785275e-10,1.365353e-12
1,OEFFGRUENFLOGD.2290,2002.715999,2002.715999,-1.271701e-09,-6.349882e-11
2,OEFFGRUENFLOGD.2291,3243.484798,3243.484798,-3.210516e-10,-9.898355e-12
3,OEFFGRUENFLOGD.2292,5086.636449,5086.636449,-3.637979e-12,-7.152032e-14
4,OEFFGRUENFLOGD.2293,4592.970413,4592.970413,0.000000e+00,0.000000e+00


In [12]:
lost_districts = set(census_boundaries['ZBEZ']) - set(green_in_census['ZBEZ'])
print(f"Districts with no green area: {lost_districts}")

Districts with no green area: {'2120', '2231'}


In [18]:
green_area_per_district = green_in_census.groupby('ZBEZ')['new_area'].sum().reset_index()
print("Sum green areas per census district - my calculation, after groupby:", green_area_per_district["new_area"].sum())

green_area_per_district = green_area_per_district.merge(census_boundaries, on='ZBEZ', how='left')
green_area_per_district['new_area'] = green_area_per_district['new_area'].fillna(0)
green_area_per_district['green_ratio'] = green_area_per_district['new_area'] / green_area_per_district['area_boundaries']
print(len(green_area_per_district))
print(green_area_per_district.isnull().sum())
green_area_per_district.head(3)

Sum green areas per census district - my calculation, after groupby: 126944693.16638355
248
ZBEZ                   0
new_area               0
original_censusarea    0
geometry_census        0
area_boundaries        0
green_ratio            0
dtype: int64


,ZBEZ,new_area,original_censusarea,geometry_census,area_boundaries,green_ratio
0,0101,1177.028363,377461.6586,"POLYGON ((3485.011 341498.136, 3486.181 341497...",377461.659643,0.003118
1,0102,77665.401597,379376.7503,"POLYGON ((3487.781 341009.076, 3521.041 341061...",379376.751348,0.204718
2,0103,25072.878241,465496.8591,"POLYGON ((2735.141 340257.126, 2714.401 340256...",465496.860336,0.053863


In [20]:
# Number of census districts where green_ratio > 1
problem_count_g = (green_area_per_district['green_ratio'] > 1).sum()
print("Districts with green_ratio > 1:", problem_count_g)

Districts with green_ratio > 1: 0


,ZBEZ,new_area,original_censusarea,geometry_census,area_boundaries,green_ratio
0,0101,1.177028e+03,3.774617e+05,"POLYGON ((3485.011 341498.136, 3486.181 341497...",3.774617e+05,0.003118
16,0210,2.168042e+05,1.250098e+06,"POLYGON ((11808.831 336679.296, 11837.011 3366...",1.250098e+06,0.173430
109,1310,2.213247e+07,2.349476e+07,"POLYGON ((-6004.259 340219.656, -5945.939 3401...",2.349476e+07,0.942017
197,2130,3.139292e+03,7.276094e+05,"POLYGON ((3650.591 349742.196, 3665.511 349724...",7.276095e+05,0.004315
227,2230,2.242157e+07,2.608180e+07,"POLYGON ((15203.561 340296.236, 15202.431 3402...",2.608180e+07,0.859663
